# TT-LoRA Core Count Run Analysis

Use this notebook to analyze runs produced by `ttlora_core_count_study`.

You can point `ANALYSIS_ROOT` at any of these:
- the phase `runs/` directory
- a single run group such as `runs/enron_reconstruction`
- the phase root `ttlora_core_count_study`
- a suite directory under `suites/`

The notebook will automatically discover `summary.json`, `history.csv`, `step_history.csv`, and any `execution_log.csv` / `ray_execution_log.csv` files it can find nearby.


In [ ]:
from __future__ import annotations

import ast
import json
import re
from pathlib import Path
from typing import Any

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError(
        'This notebook needs matplotlib in the active kernel. Install it with something like `pip install matplotlib seaborn pandas`. '
        'If you already have a project notebook environment, switch the kernel to that environment first.'
    ) from exc

import pandas as pd

try:
    import seaborn as sns
except ImportError as exc:
    raise ImportError(
        'This notebook uses seaborn for plotting. Install it with something like `pip install seaborn` in the active kernel.'
    ) from exc

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 120)

# Update this to the path you want to analyze.
ANALYSIS_ROOT = Path('../runs').resolve()

# If True, only keep runs with returncode == 0 when execution logs are available.
SHOW_ONLY_SUCCESSFUL = True

ANALYSIS_ROOT


In [ ]:
def _safe_parse(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (dict, list, int, float, bool)):
        return value
    text = str(value).strip()
    if text == '' or text.lower() in {'nan', 'none', 'null'}:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            pass
    return value


def _maybe_number(value: Any) -> Any:
    if value is None or isinstance(value, (dict, list, bool)):
        return value
    if isinstance(value, (int, float)):
        return value
    text = str(value).strip()
    if text == '':
        return None
    try:
        if re.fullmatch(r'-?\d+', text):
            return int(text)
        return float(text)
    except ValueError:
        return value


def _phase_root_from(path: Path) -> Path | None:
    for candidate in (path, *path.parents):
        if candidate.name == 'ttlora_core_count_study':
            return candidate
    return None


def discover_summary_paths(root: Path) -> list[Path]:
    root = Path(root).expanduser().resolve()
    if root.is_file() and root.name == 'summary.json':
        return [root]

    phase_root = _phase_root_from(root)
    if phase_root is not None and 'runs' not in root.parts and (phase_root / 'runs').exists():
        search_root = phase_root / 'runs'
    else:
        search_root = root

    if search_root.is_file():
        return []
    return sorted({path.resolve() for path in search_root.rglob('summary.json')})


def discover_execution_logs(root: Path) -> list[Path]:
    root = Path(root).expanduser().resolve()
    phase_root = _phase_root_from(root)

    if root.is_dir() and root.parent.name == 'suites':
        search_root = root
    elif phase_root is not None and (phase_root / 'suites').exists():
        search_root = phase_root / 'suites'
    else:
        search_root = root

    if not search_root.exists() or search_root.is_file():
        return []

    paths: set[Path] = set()
    for pattern in ('execution_log.csv', 'ray_execution_log.csv'):
        paths.update(path.resolve() for path in search_root.rglob(pattern))
    return sorted(paths)


def _extract_total_cores(row: dict[str, Any]) -> int | None:
    ttlora_shape = row.get('ttlora_shape')
    if isinstance(ttlora_shape, list) and ttlora_shape:
        return len(ttlora_shape)

    weight_configs = row.get('ttlora_weight_configs')
    if isinstance(weight_configs, list) and weight_configs:
        tt_shape = weight_configs[0].get('tt_shape')
        if isinstance(tt_shape, list) and tt_shape:
            return len(tt_shape)

    run_name = str(row.get('run_name', ''))
    match = re.search(r'cores(\d+)', run_name)
    return int(match.group(1)) if match else None


def _extract_input_output_factors(row: dict[str, Any]) -> tuple[list[int] | None, list[int] | None]:
    input_factors = row.get('ttlora_input_factors')
    output_factors = row.get('ttlora_output_factors')
    if isinstance(input_factors, list) or isinstance(output_factors, list):
        return input_factors, output_factors

    weight_configs = row.get('ttlora_weight_configs')
    if isinstance(weight_configs, list) and weight_configs:
        return weight_configs[0].get('input_factors'), weight_configs[0].get('output_factors')

    return None, None


def _metric_preferences(row: dict[str, Any]) -> list[tuple[str, str]]:
    task_type = str(row.get('task_type', '')).lower()
    if task_type == 'generation':
        return [
            ('best_validation_token_accuracy', 'max'),
            ('best_validation_perplexity', 'min'),
            ('best_validation_loss', 'min'),
        ]
    return [
        ('best_validation_accuracy', 'max'),
        ('best_validation_loss', 'min'),
        ('final_validation_accuracy', 'max'),
        ('final_validation_loss', 'min'),
    ]


def _attach_primary_metric(row: dict[str, Any]) -> None:
    row['primary_metric_name'] = None
    row['primary_metric_value'] = None
    row['primary_metric_mode'] = None
    for metric_name, mode in _metric_preferences(row):
        value = row.get(metric_name)
        if isinstance(value, (int, float)):
            row['primary_metric_name'] = metric_name
            row['primary_metric_value'] = value
            row['primary_metric_mode'] = mode
            return


def load_summary_frame(root: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for summary_path in discover_summary_paths(root):
        payload = json.loads(summary_path.read_text())
        row = {key: _safe_parse(value) for key, value in payload.items()}
        row['summary_path'] = str(summary_path)
        row['run_name'] = summary_path.parent.name
        row['run_group'] = summary_path.parent.parent.name
        row['resolved_run_dir'] = str(Path(row.get('resolved_run_dir') or summary_path.parent))

        ttlora_shape = row.get('ttlora_shape')
        if not isinstance(ttlora_shape, list):
            weight_configs = row.get('ttlora_weight_configs')
            if isinstance(weight_configs, list) and weight_configs:
                row['ttlora_shape'] = weight_configs[0].get('tt_shape')

        input_factors, output_factors = _extract_input_output_factors(row)
        row['resolved_input_factors'] = input_factors
        row['resolved_output_factors'] = output_factors
        row['input_cores'] = len(input_factors) if isinstance(input_factors, list) else None
        row['output_cores'] = len(output_factors) if isinstance(output_factors, list) else None
        row['total_cores'] = _extract_total_cores(row)

        trainable = row.get('trainable_parameters')
        total = row.get('total_parameters')
        if isinstance(trainable, (int, float)) and isinstance(total, (int, float)) and total:
            row['trainable_parameter_fraction'] = trainable / total
            row['trainable_parameter_percent'] = 100.0 * trainable / total
        else:
            row['trainable_parameter_fraction'] = None
            row['trainable_parameter_percent'] = None

        _attach_primary_metric(row)
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    for column in df.columns:
        if df[column].map(lambda value: isinstance(value, (list, dict))).any():
            continue
        df[column] = df[column].map(_maybe_number)
    return df


def load_execution_log_frame(root: Path) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for log_path in discover_execution_logs(root):
        frame = pd.read_csv(log_path)
        frame['execution_log_path'] = str(log_path)
        frame['execution_log_name'] = log_path.name
        frame['execution_suite_name'] = log_path.parent.name
        frames.append(frame)

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    for column in df.columns:
        if column in {'generation_weight_candidates', 'input_factors', 'output_factors', 'tt_shape'}:
            df[column] = df[column].map(_safe_parse)
        else:
            df[column] = df[column].map(_maybe_number)

    if 'run_name' in df.columns:
        df = df.sort_values('execution_log_path').drop_duplicates(subset=['run_name'], keep='last')
    return df


def build_run_table(root: Path) -> pd.DataFrame:
    summary_df = load_summary_frame(root)
    log_df = load_execution_log_frame(root)

    if summary_df.empty and log_df.empty:
        raise FileNotFoundError(f'No summaries or execution logs found under {Path(root).resolve()}')

    if summary_df.empty:
        df = log_df.copy()
        df['has_summary'] = False
    elif log_df.empty:
        df = summary_df.copy()
        df['has_summary'] = True
    else:
        log_columns = [column for column in log_df.columns if column not in summary_df.columns or column == 'run_name']
        df = summary_df.merge(log_df[log_columns], on='run_name', how='outer')
        df['has_summary'] = df['summary_path'].notna()

    if SHOW_ONLY_SUCCESSFUL and 'returncode' in df.columns:
        keep_mask = df['returncode'].isna() | (df['returncode'] == 0)
        df = df.loc[keep_mask].copy()

    sort_columns = [column for column in ['dataset_name', 'ttlora_variant', 'total_cores', 'seed'] if column in df.columns]
    if sort_columns:
        df = df.sort_values(sort_columns).reset_index(drop=True)
    return df


def metric_candidates(df: pd.DataFrame) -> list[str]:
    preferred = [
        'best_validation_accuracy',
        'best_validation_token_accuracy',
        'best_validation_loss',
        'best_validation_perplexity',
        'final_validation_accuracy',
        'final_validation_token_accuracy',
        'final_validation_loss',
        'final_validation_perplexity',
        'avg_epoch_seconds',
        'elapsed_seconds',
        'max_peak_memory_gb',
        'trainable_parameters',
    ]
    return [column for column in preferred if column in df.columns]


def choose_best_row(df: pd.DataFrame, metric: str) -> pd.Series:
    valid = df.dropna(subset=[metric]).copy()
    if valid.empty:
        raise ValueError(f'No rows contain metric {metric!r}')
    minimize = metric.endswith('loss') or metric.endswith('perplexity') or metric in {'elapsed_seconds', 'avg_epoch_seconds', 'max_peak_memory_gb'}
    return valid.sort_values(metric, ascending=minimize).iloc[0]


def load_history(row: pd.Series) -> pd.DataFrame:
    history_path = row.get('history_path')
    if not history_path or not Path(history_path).exists():
        raise FileNotFoundError(f'History file not found for run {row.get("run_name")}')
    history_df = pd.read_csv(history_path)
    for column in history_df.columns:
        history_df[column] = history_df[column].map(_maybe_number)
    return history_df


In [ ]:
runs_df = build_run_table(ANALYSIS_ROOT)

print(f'Loaded {len(runs_df)} runs from {ANALYSIS_ROOT}')
print('Datasets:', sorted(runs_df['dataset_name'].dropna().astype(str).unique()) if 'dataset_name' in runs_df else [])
print('Variants:', sorted(runs_df['ttlora_variant'].dropna().astype(str).unique()) if 'ttlora_variant' in runs_df else [])
print('Metric candidates:', metric_candidates(runs_df))

display_columns = [
    column
    for column in [
        'run_name',
        'dataset_name',
        'ttlora_variant',
        'task_type',
        'total_cores',
        'learning_rate',
        'seed',
        'primary_metric_name',
        'primary_metric_value',
        'best_validation_accuracy',
        'best_validation_token_accuracy',
        'best_validation_loss',
        'best_validation_perplexity',
        'trainable_parameters',
        'avg_epoch_seconds',
        'elapsed_seconds',
        'max_peak_memory_gb',
        'returncode',
    ]
    if column in runs_df.columns
]
runs_df[display_columns].head(20)


In [ ]:
# Adjust these filters and rerun this cell.
DATASET_FILTER = None          # e.g. 'enron', 'ptb', 'mrpc'
VARIANT_FILTER = None          # e.g. 'reconstruction', 'contraction'
METRIC = 'best_validation_loss'

plot_df = runs_df.copy()
if DATASET_FILTER is not None and 'dataset_name' in plot_df.columns:
    plot_df = plot_df.loc[plot_df['dataset_name'] == DATASET_FILTER].copy()
if VARIANT_FILTER is not None and 'ttlora_variant' in plot_df.columns:
    plot_df = plot_df.loc[plot_df['ttlora_variant'] == VARIANT_FILTER].copy()

if plot_df.empty:
    raise ValueError('The current filters removed every run. Update DATASET_FILTER / VARIANT_FILTER.')
if METRIC not in plot_df.columns:
    raise KeyError(f'Metric {METRIC!r} is not in the dataframe. Pick one from: {metric_candidates(plot_df)}')

metric_df = plot_df.dropna(subset=['total_cores', METRIC]).copy()
if metric_df.empty:
    raise ValueError(f'No runs contain both total_cores and {METRIC!r}.')

fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)

sns.lineplot(
    data=metric_df,
    x='total_cores',
    y=METRIC,
    hue='dataset_name' if 'dataset_name' in metric_df.columns else None,
    style='ttlora_variant' if 'ttlora_variant' in metric_df.columns else None,
    marker='o',
    estimator='mean',
    errorbar='sd',
    ax=axes[0, 0],
)
axes[0, 0].set_title(f'{METRIC} vs total cores')

if 'trainable_parameters' in metric_df.columns:
    sns.lineplot(
        data=metric_df.dropna(subset=['trainable_parameters']),
        x='total_cores',
        y='trainable_parameters',
        hue='dataset_name' if 'dataset_name' in metric_df.columns else None,
        style='ttlora_variant' if 'ttlora_variant' in metric_df.columns else None,
        marker='o',
        estimator='mean',
        errorbar='sd',
        ax=axes[0, 1],
    )
    axes[0, 1].set_title('Trainable parameters vs total cores')
else:
    axes[0, 1].axis('off')

if 'avg_epoch_seconds' in metric_df.columns:
    sns.lineplot(
        data=metric_df.dropna(subset=['avg_epoch_seconds']),
        x='total_cores',
        y='avg_epoch_seconds',
        hue='dataset_name' if 'dataset_name' in metric_df.columns else None,
        style='ttlora_variant' if 'ttlora_variant' in metric_df.columns else None,
        marker='o',
        estimator='mean',
        errorbar='sd',
        ax=axes[1, 0],
    )
    axes[1, 0].set_title('Average epoch time vs total cores')
else:
    axes[1, 0].axis('off')

if 'max_peak_memory_gb' in metric_df.columns:
    sns.lineplot(
        data=metric_df.dropna(subset=['max_peak_memory_gb']),
        x='total_cores',
        y='max_peak_memory_gb',
        hue='dataset_name' if 'dataset_name' in metric_df.columns else None,
        style='ttlora_variant' if 'ttlora_variant' in metric_df.columns else None,
        marker='o',
        estimator='mean',
        errorbar='sd',
        ax=axes[1, 1],
    )
    axes[1, 1].set_title('Peak memory vs total cores')
else:
    axes[1, 1].axis('off')

plt.show()

group_columns = [column for column in ['dataset_name', 'ttlora_variant'] if column in metric_df.columns]
if group_columns:
    ascending = METRIC.endswith('loss') or METRIC.endswith('perplexity')
    best_runs = (
        metric_df.sort_values(METRIC, ascending=ascending)
        .groupby(group_columns, dropna=False)
        .head(5)
    )
    display(best_runs[[column for column in display_columns if column in best_runs.columns]])


In [ ]:
# Pick a run explicitly, or leave RUN_NAME = None to automatically choose the best run for METRIC.
RUN_NAME = None

history_source = plot_df.copy()
if RUN_NAME is not None:
    selected = history_source.loc[history_source['run_name'] == RUN_NAME]
    if selected.empty:
        raise ValueError(f'Run {RUN_NAME!r} was not found in the filtered dataframe.')
    selected_row = selected.iloc[0]
else:
    selected_row = choose_best_row(history_source, METRIC)

selected_summary = selected_row[[column for column in display_columns if column in selected_row.index]]
display(selected_summary.to_frame(name='value'))

history_df = load_history(selected_row)
history_columns = history_df.columns.tolist()
print('History columns:', history_columns)

fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)

sns.lineplot(data=history_df, x='epoch', y='train_loss', label='train_loss', ax=axes[0])
if 'validation_loss' in history_df.columns:
    sns.lineplot(data=history_df, x='epoch', y='validation_loss', label='validation_loss', ax=axes[0])
axes[0].set_title('Loss curves')

metric_column = None
for candidate in ['validation_accuracy', 'validation_token_accuracy', 'train_accuracy', 'train_token_accuracy']:
    if candidate in history_df.columns:
        metric_column = candidate
        break
if metric_column is not None:
    sns.lineplot(data=history_df, x='epoch', y=metric_column, ax=axes[1])
    axes[1].set_title(metric_column)
else:
    axes[1].axis('off')

if 'avg_grad_norm' in history_df.columns:
    sns.lineplot(data=history_df, x='epoch', y='avg_grad_norm', label='avg_grad_norm', ax=axes[2])
    if 'max_grad_norm' in history_df.columns:
        sns.lineplot(data=history_df, x='epoch', y='max_grad_norm', label='max_grad_norm', ax=axes[2])
    axes[2].set_title('Gradient norm trends')
else:
    axes[2].axis('off')

plt.show()

if 'step_history_path' in selected_row.index and selected_row['step_history_path'] and Path(selected_row['step_history_path']).exists():
    step_history_df = pd.read_csv(selected_row['step_history_path'])
    for column in step_history_df.columns:
        step_history_df[column] = step_history_df[column].map(_maybe_number)
    display(step_history_df.head())
else:
    print('No step_history.csv found for the selected run.')
